In [2]:
import numpy as np
import pandas as pd
import pandas.api.types
import sklearn.metrics

In [7]:
y_train = pd.read_csv('/kaggle/input/rsna-2023-abdominal-trauma-detection/train.csv')
y_train.head()
Injuries = ['bowel_healthy', 'bowel_injury', 
            'extravasation_healthy', 'extravasation_injury', 
            'kidney_healthy', 'kidney_low', 'kidney_high', 
            'liver_healthy', 'liver_low', 'liver_high', 
            'spleen_healthy', 'spleen_low', 'spleen_high', 
            'any_injury']

**This function takes a DataFrame (df) and a list of column names (group_columns) as input.
It calculates row totals for the specified group of columns.
Then it normalizes the probabilities in each row by dividing each value by its row total.
This function ensures that the sum of probabilities in each row is equal to 1 after normalization.
It returns the modified DataFrame.
This function wraps the log_loss function from sklearn.metrics
It calculates the log loss for the "any_injury" label by taking the maximum probability of injury across all target categories.
Finally, it returns the mean of all label group losses, including the "any_injury" loss, as the custom score.**

In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import log_loss

def normalize_probabilities_to_one(df: pd.DataFrame, group_columns: list) -> pd.DataFrame:
    # Calculate row totals
    row_totals = df[group_columns].sum(axis=1)

    # Check if any row has zero total
    if (row_totals == 0).any():
        raise ValueError('All rows must contain at least one non-zero prediction')

    # Normalize the probabilities
    df[group_columns] = df[group_columns].div(row_totals, axis=0)
    return df

def custom_log_loss(y_true, y_pred, sample_weight=None):
    # Implement a custom log loss function if needed
    # You can enhance this function for your specific requirements
    return log_loss(y_true, y_pred, sample_weight=sample_weight)

def custom_score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    # Drop row_id_column from the DataFrames
    solution = solution.drop(columns=[row_id_column_name])
    submission = submission.drop(columns=[row_id_column_name])

    # Check if submission values are numeric
    if not pd.api.types.is_numeric_dtype(submission.values):
        raise ValueError('All submission values must be numeric')

    # Check if submission values are finite
    if not np.isfinite(submission.values).all():
        raise ValueError('All submission values must be finite')

    # Check if labels and predictions are non-negative
    if solution.min().min() < 0:
        raise ValueError('All labels must be at least zero')
    if submission.min().min() < 0:
        raise ValueError('All predictions must be at least zero')

    # Define your target categories
    binary_targets = ['bowel', 'extravasation']
    triple_level_targets = ['kidney', 'liver', 'spleen']
    all_target_categories = binary_targets + triple_level_targets

    label_group_losses = []

    for category in all_target_categories:
        if category in binary_targets:
            col_group = [f'{category}_healthy', f'{category}_injury']
        else:
            col_group = [f'{category}_healthy', f'{category}_low', f'{category}_high']

        solution = normalize_probabilities_to_one(solution, col_group)

        for col in col_group:
            if col not in submission.columns:
                raise ValueError(f'Missing submission column {col}')

        submission = normalize_probabilities_to_one(submission, col_group)

        label_group_losses.append(
            custom_log_loss(
                y_true=solution[col_group].values,
                y_pred=submission[col_group].values,
                sample_weight=solution[f'{category}_weight'].values
            )
        )

    # Calculate the custom score for "any_injury" label
    healthy_cols = [x + '_healthy' for x in all_target_categories]
    any_injury_labels = (1 - solution[healthy_cols]).max(axis=1)
    any_injury_predictions = (1 - submission[healthy_cols]).max(axis=1)

    any_injury_loss = custom_log_loss(
        y_true=any_injury_labels.values,
        y_pred=any_injury_predictions.values,
        sample_weight=solution['any_injury_weight'].values
    )

    label_group_losses.append(any_injury_loss)

    # Calculate the mean of label group losses
    return np.mean(label_group_losses)


**The function first creates a copy of the y_train DataFrame to avoid modifying the original data.
For each target category, it adds a corresponding weight column to the solution DataFrame.
For binary targets (bowel_injury and extravasation_injury), it assigns weights of 2 for positive instances and 1 for negative instances.
For triple-level targets (kidney, liver, and spleen), it assigns weights of 2 for low, 4 for high, and 1 for healthy instances.
For the "any_injury" label, it assigns a weight of 6 for positive instances and 1 for negative instances.**

In [11]:
def create_training_solution(y_train):
    sol_train = y_train.copy()
    sol_train['bowel_weight'] = np.where(sol_train['bowel_injury'] == 1, 2, 1)
    sol_train['extravasation_weight'] = np.where(sol_train['extravasation_injury'] == 1, 6, 1)
    sol_train['kidney_weight'] = np.where(sol_train['kidney_low'] == 1, 2, np.where(sol_train['kidney_high'] == 1, 4, 1))
    sol_train['liver_weight'] = np.where(sol_train['liver_low'] == 1, 2, np.where(sol_train['liver_high'] == 1, 4, 1))
    sol_train['spleen_weight'] = np.where(sol_train['spleen_low'] == 1, 2, np.where(sol_train['spleen_high'] == 1, 4, 1))
    sol_train['any_injury_weight'] = np.where(sol_train['any_injury'] == 1, 6, 1)
    return sol_train

In [12]:
solution_train = create_training_solution(y_train)

y_pred = y_train.copy()
y_pred[Injuries] = y_train[Injuries].mean().tolist()

custom_score_value = custom_score(solution_train, y_pred, 'patient_id')
print(f'Training custom score without scaling: {custom_score_value}')

Training custom score without scaling: 0.7860663285561644


In [14]:
scale_by_2 = ['kidney_low', 'liver_low', 'spleen_low', 'spleen_high']
scale_by_4 = ['bowel_injury', 'kidney_high', 'liver_high']
scale_by_6 = ['extravasation_injury', 'any_injury']
scale_healthy = ['bowel_healthy', 'extravasation_healthy', 'kidney_healthy', 'liver_healthy', 'spleen_healthy']

# Adjust the scaling factors
sf_2 = 5.0  # Adjusted
sf_4 = 10.0  # Adjusted
sf_6 = 40.0  # Adjusted
scale_h = 1.2

# Use the create_training_solution function with adjusted weights
solution_train = create_training_solution(y_train)

# Create mean predictions
y_pred = y_train.copy()
y_pred[Injuries] = y_train[Injuries].mean().tolist()

# Adjust the scaling of the predictions
y_pred[scale_by_2] *= sf_2
y_pred[scale_by_4] *= sf_4
y_pred[scale_by_6] *= sf_6
y_pred[scale_healthy] *= scale_h

# Calculate the score with adjusted weights and scaling using the custom_score function
adjusted_score = custom_score(solution_train, y_pred, 'patient_id')
print(f'Training score with adjustments: {adjusted_score}')


Training score with adjustments: 0.6395110368338588


In [16]:
solution_train = create_training_solution(y_train)

y_pred = y_train.copy()
y_pred[Injuries] = y_train[Injuries].mean().tolist()

# Scale each target
y_pred[scale_by_2] *= sf_2
y_pred[scale_by_4] *= sf_4
y_pred[scale_by_6] *= sf_6
y_pred[scale_healthy] *= scale_h

improved_scale_score = custom_score(solution_train, y_pred, 'patient_id')
print(f'Training score with better scaling: {improved_scale_score}')


Training score with better scaling: 0.6395110368338588


In [18]:
submission = pd.read_csv('/kaggle/input/rsna-2023-abdominal-trauma-detection/sample_submission.csv')

submission[Injuries] = y_train[Injuries].mean().tolist()

submission[scale_by_2] *=sf_2
submission[scale_by_4] *=sf_4
submission[scale_by_6] *=sf_6
submission[scale_healthy] *=scale_h

# Save Submission!
submission.to_csv('submission.csv', index=False)
print("done")



done
